In [1]:
"""
NetCDF -> Multi-band GeoTIFF Conversion for Earth Engine Upload
------------------------------------------------------------------
Converts each variable in your consolidated NetCDF files into ONE
multi-band GeoTIFF (one band per month, 312 bands), so the whole
dataset becomes just 10 files -- small enough to drag-and-drop upload
directly in the Earth Engine Code Editor's Assets tab. No Google Cloud
Storage bucket or billing account needed.

Band naming: each band is named/described with its date as "YYYY_MM"
(e.g. "2015_06"), so the GEE App can select the right band later based
on a year/month dropdown.

HOW TO RUN (Google Colab, free):
1. Mount Drive, pip install rasterio if needed:
       !pip install rasterio --quiet
2. Edit the CONFIG section below.
3. Run the whole script.
4. Download the resulting .tif files from Drive to your computer (or
   Colab lets you download directly), then upload them via:
   Earth Engine Code Editor -> Assets tab -> NEW -> Image Upload
   -> drag and drop each .tif file.
"""

import os
import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_bounds

# ---- CONFIG ----
BASE_DIR = "D:\\Project Works\\Drought_QLD\\Data\\Final DB for Project\\processed"
SPI_FILE = os.path.join(BASE_DIR, "QLD_SPI_2000_2025.nc")
SPEI_FILE = os.path.join(BASE_DIR, "QLD_SPEI_2000_2025.nc")
PD_FILE = os.path.join(BASE_DIR, "QLD_protracted_drought_label_2000_2025.nc")
OUTPUT_DIR = os.path.join(BASE_DIR, "geotiffs_for_gee")
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ------------------


def get_transform(lat, lon):
    """Build an affine transform from pixel-center lat/lon coordinate arrays."""
    res_lat = abs(float(lat[1] - lat[0]))
    res_lon = abs(float(lon[1] - lon[0]))
    west = float(lon.min()) - res_lon / 2
    east = float(lon.max()) + res_lon / 2
    south = float(lat.min()) - res_lat / 2
    north = float(lat.max()) + res_lat / 2
    return from_bounds(west, south, east, north, len(lon), len(lat))


def write_multiband_geotiff(data_array, out_path, band_dates=None):
    """Write an xarray DataArray with dims (time, lat, lon) -- or (lat, lon)
    for a static layer -- to a single multi-band GeoTIFF."""
    lat = data_array["lat"].values
    lon = data_array["lon"].values
    transform = get_transform(lat, lon)

    # Ensure lat is ascending in the array, then flip for GeoTIFF (north-up expected)
    data_array = data_array.sortby("lat")

    if "time" in data_array.dims:
        values = data_array.transpose("time", "lat", "lon").values  # (bands, lat, lon)
        values = np.flip(values, axis=1)  # flip lat so row 0 = north, matches transform
        n_bands = values.shape[0]
    else:
        values = data_array.values[np.newaxis, :, :]
        values = np.flip(values, axis=1)
        n_bands = 1

    values = np.nan_to_num(values, nan=-9999).astype("float32")

    with rasterio.open(
        out_path, "w", driver="GTiff",
        height=values.shape[1], width=values.shape[2],
        count=n_bands, dtype="float32",
        crs="EPSG:4326", transform=transform,
        nodata=-9999, compress="LZW",
    ) as dst:
        for b in range(n_bands):
            dst.write(values[b], b + 1)
            if band_dates is not None:
                band_label = band_dates[b]
                dst.set_band_description(b + 1, band_label)

    print(f"  Saved: {out_path} ({n_bands} bands, "
          f"{os.path.getsize(out_path) / 1e6:.1f} MB)")


def month_labels(time_coord):
    return [str(t)[:7].replace("-", "_") for t in time_coord.dt.strftime("%Y-%m").values]


def main():
    print("Converting SPI...")
    spi_ds = xr.open_dataset(SPI_FILE)
    labels = month_labels(spi_ds["time"])
    for var in ["spi_01", "spi_03", "spi_06", "spi_12"]:
        scale = var.split("_")[1].lstrip("0") or "0"
        out_path = os.path.join(OUTPUT_DIR, f"QLD_SPI_scale{scale}.tif")
        write_multiband_geotiff(spi_ds[var], out_path, labels)
    spi_ds.close()

    print("\nConverting SPEI...")
    spei_ds = xr.open_dataset(SPEI_FILE)
    labels = month_labels(spei_ds["time"])
    for var in ["spei_gamma_01", "spei_gamma_03", "spei_gamma_06", "spei_gamma_12"]:
        scale = var.split("_")[-1].lstrip("0") or "0"
        out_path = os.path.join(OUTPUT_DIR, f"QLD_SPEI_scale{scale}.tif")
        write_multiband_geotiff(spei_ds[var], out_path, labels)
    spei_ds.close()

    print("\nConverting Protracted Drought...")
    pd_ds = xr.open_dataset(PD_FILE)
    labels = month_labels(pd_ds["time"])
    out_path = os.path.join(OUTPUT_DIR, "QLD_protracted_drought.tif")
    write_multiband_geotiff(pd_ds["protracted_drought"], out_path, labels)

    print("\nConverting Bioregion ID (static, single band)...")
    out_path = os.path.join(OUTPUT_DIR, "QLD_bioregion_id.tif")
    write_multiband_geotiff(pd_ds["bioregion_id"], out_path, band_dates=["bioregion_id"])
    pd_ds.close()

    print(f"\nDone. 10 GeoTIFFs saved to: {OUTPUT_DIR}")
    print("Next: download these files and upload them in the Earth Engine "
          "Code Editor's Assets tab (NEW -> Image Upload).")


if __name__ == "__main__":
    main()

Converting SPI...
  Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\processed\geotiffs_for_gee\QLD_SPI_scale1.tif (312 bands, 75.4 MB)
  Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\processed\geotiffs_for_gee\QLD_SPI_scale3.tif (312 bands, 91.1 MB)
  Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\processed\geotiffs_for_gee\QLD_SPI_scale6.tif (312 bands, 92.4 MB)
  Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\processed\geotiffs_for_gee\QLD_SPI_scale12.tif (312 bands, 90.3 MB)

Converting SPEI...
  Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\processed\geotiffs_for_gee\QLD_SPEI_scale1.tif (312 bands, 100.6 MB)
  Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\processed\geotiffs_for_gee\QLD_SPEI_scale3.tif (312 bands, 100.8 MB)
  Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\processed\geotiffs_for_gee\QLD_SPEI_scale6.tif (312 bands, 100.7 MB)
  Saved: D:\Project Works\Drought_QLD\